<a href="https://colab.research.google.com/github/Anmolsharma44/AI-LLM-IITJammu/blob/main/Encoder/Encoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np

# 1. The input sentence
sentence = "The animal didn't cross the street because it was too tired."
tokens = sentence.replace(".", "").split()  #that .replace is for the . at the end of sentence
seq_len = len(tokens)
embed_dim = 4 # Small embedding dimension so we can print it easily

# 2. Mock Vocabulary Lookup (Creating a random static embedding for each unique word)
np.random.seed(42) # For reproducible vectors
vocab = {word: np.random.randn(embed_dim) for word in set(tokens)}

# 3. Create the input matrix X (Sequence length x Embedding Dimension)
X = np.array([vocab[token] for token in tokens])

print(f"--- STEP 1: TOKENIZER & INITIAL EMBEDDINGS ---")
print(f"Tokens: {tokens}")
print(f"Shape of Input Matrix X: {X.shape} (Sequence Length: {seq_len}, Embed Dim: {embed_dim})\n")
print("Initial Vector for 'it':")
it_index = tokens.index("it")
print(X[it_index])

--- STEP 1: TOKENIZER & INITIAL EMBEDDINGS ---
Tokens: ['The', 'animal', "didn't", 'cross', 'the', 'street', 'because', 'it', 'was', 'too', 'tired']
Shape of Input Matrix X: (11, 4) (Sequence Length: 11, Embed Dim: 4)

Initial Vector for 'it':
[ 0.73846658  0.17136828 -0.11564828 -0.3011037 ]


In [ ]:
# Create random projection weight matrices
W_Q = np.random.randn(embed_dim, embed_dim) #its dimensions is 4*4
W_K = np.random.randn(embed_dim, embed_dim)
W_V = np.random.randn(embed_dim, embed_dim)

# Calculate Q, K, V matrices via matrix multiplication (@ operator)
Q = X @ W_Q # Now its dimension beocme 11*4,4*4 ie 11*4
K = X @ W_K
V = X @ W_V

print(f"--- STEP 2: GENERATING QUERIES, KEYS, VALUES ---")
print(f"Shape of Q, K, and V matrices: {Q.shape}\n")
print(f"Vectors for the target word 'it' (Index {it_index}):")
print(f"Query vector: {Q[it_index]}")
print(f"Key vector:   {K[it_index]}")
print(f"Value vector: {V[it_index]}")

--- STEP 2: GENERATING QUERIES, KEYS, VALUES ---
Shape of Q, K, and V matrices: (11, 4)

Vectors for the target word 'it' (Index 7):
Query vector: [-0.70197744 -0.81134437 -0.50360659  0.31321736]
Key vector:   [-0.24564824 -0.30119942 -0.0823115  -1.13673408]
Value vector: [ 0.27965021 -0.25612855  0.18593134 -1.88625038]


In [ ]:
#finding the similarity score
def softmax(x):
    # Stabilized softmax function for NumPy arrays
    exp_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)

# 1. Score calculation (Q multiplied by Transposed K)
raw_scores = Q @ K.T

# 2. Scale by the square root of key dimension
scaled_scores = raw_scores / np.sqrt(embed_dim)

# 3. Apply Softmax to get structural weights (percentages)
attention_weights = softmax(scaled_scores)

print(f"--- STEP 3: ATTENTION WEIGHTS ($w_{{ij}}$) ---")
print(f"Attention Matrix Shape: {attention_weights.shape} ({seq_len}x{seq_len})\n")

print(f"Let's look at what the word 'it' is focusing on:")
for i, token in enumerate(tokens):
    weight_from_it = attention_weights[it_index][i]
    print(f"  'it' attends to '{token}': {weight_from_it:.4f}")

--- STEP 3: ATTENTION WEIGHTS ($w_{ij}$) ---
Attention Matrix Shape: (11, 11) (11x11)

Let's look at what the word 'it' is focusing on:
  'it' attends to 'The': 0.1287
  'it' attends to 'animal': 0.0951
  'it' attends to 'didn't': 0.0220
  'it' attends to 'cross': 0.0302
  'it' attends to 'the': 0.1726
  'it' attends to 'street': 0.1332
  'it' attends to 'because': 0.1198
  'it' attends to 'it': 0.0656
  'it' attends to 'was': 0.0370
  'it' attends to 'too': 0.1249
  'it' attends to 'tired': 0.0709


In [ ]:
# Compute the linear combination of the attention matrix and the values
dynamic_embeddings = attention_weights @ V

print(f"--- STEP 4: WEIGHTED AVERAGE EMBEDDING ---")
print(f"Shape of Context-Aware Matrix: {dynamic_embeddings.shape}\n")
print("Dynamic Embedding for the word 'it' (now containing pieces of other words):")
print(dynamic_embeddings[it_index])

--- STEP 4: WEIGHTED AVERAGE EMBEDDING ---
Shape of Context-Aware Matrix: (11, 4)

Dynamic Embedding for the word 'it' (now containing pieces of other words):
[ 0.33655442 -0.2174334  -1.31582715 -0.01156147]


In [ ]:
# Simple Feed-Forward Network: Linear -> ReLU -> Linear
W_ff1 = np.random.randn(embed_dim, embed_dim * 2) # Expanding dimension
b_ff1 = np.zeros(embed_dim * 2)
W_ff2 = np.random.randn(embed_dim * 2, embed_dim) # Contracting back
b_ff2 = np.zeros(embed_dim)

# First linear layer + ReLU activation
hidden_ffn = np.maximum(0, (dynamic_embeddings @ W_ff1) + b_ff1)

# Second linear layer to get the final structural encodings
encoder_hidden_states = (hidden_ffn @ W_ff2) + b_ff2

print(f"--- STEP 5: FEED-FORWARD NETWORK OUTPUT ---")
print(f"Final Encoder Output Shape: {encoder_hidden_states.shape}\n")
print("This matrix represents the final 'Hidden State' passed to subsequent layers or the decoder:")
print(encoder_hidden_states)

--- STEP 5: FEED-FORWARD NETWORK OUTPUT ---
Final Encoder Output Shape: (11, 4)

This matrix represents the final 'Hidden State' passed to subsequent layers or the decoder:
[[-1.83089791  4.78071984 -3.18333633  5.97014338]
 [-2.13981592  2.80310709 -1.93814336  3.71008262]
 [-6.04461871  2.42833893 -6.18175092  8.94347786]
 [-3.78972242  1.06434751 -5.01372734  5.06083537]
 [-2.29400051  5.23024507 -3.3211747   6.26971402]
 [-2.32986762  5.37587381 -3.46352338  6.50553868]
 [-0.20486026  1.16443548 -1.1469971   1.17009413]
 [-2.58723876  1.17615999 -2.47452355  3.81861725]
 [-4.34128876  1.6379633  -4.81807164  6.47271629]
 [-1.66552877  4.09135501 -1.023188   -2.68060819]
 [-1.16417114  1.36014664 -0.92651166  0.85574839]]
